## RAG (Retrieval-Augmented Generation)

<div style="text-align: right"> <b>KMYU</b></div>
<div style="text-align: right"> Initial issue : 2026.06.23 </div>
<div style="text-align: right"> last update : 2026.06.23 </div>

개정 이력  
- `2026.06.23` : 노트북 최초 작성

### RAG란?

LLM은 똑똑하지만 두 가지 약점이 있습니다.

1. **모르는 것을 그럴듯하게 지어낸다(환각, hallucination)**
2. **학습하지 않은 최신·내부·전문 문서** 는 알지 못한다

**RAG(검색 증강 생성)** 는 질문과 관련된 문서를 **먼저 검색해서 프롬프트에 넣어 준 다음** 답하게 하는 방법입니다. LLM이 자기 기억이 아니라 *건네받은 문서* 를 근거로 답하므로, 환각이 줄고 출처를 밝힐 수 있습니다.

런타임 흐름은 다음과 같습니다.

> 질문 → **① 검색(Retriever)** → **② 프롬프트** (검색 결과를 context로 삽입) → **③ LLM** → **④ 답변**

이 노트북에서는 `2.1_vectordb.ipynb` 에서 저장한 FAISS 인덱스를 불러와 위 흐름을 구성합니다. 특히 **프롬프트 설계** 에 집중합니다 — RAG의 품질은 검색만큼이나 *어떻게 묻고 어떻게 답하라고 지시하는가* 에 달려 있기 때문입니다.

In [1]:
import os

from dotenv import load_dotenv

load_dotenv()

True

In [2]:
print("LLM       :", os.getenv("ANTHROPIC_MODEL"))
print("Embedding :", os.getenv("VOYAGE_EMBEDDING_MODEL"))

LLM       : claude-haiku-4-5
Embedding : voyage-4-lite


### 1. 저장된 벡터 DB 로드

`2.1` 에서 저장한 FAISS 인덱스를 불러옵니다.

주의할 점이 두 가지 있습니다.
- **임베딩 모델은 저장할 때와 같은 것** 을 써야 합니다. 질문도 같은 방식으로 벡터화해야 거리 비교가 의미를 가지기 때문입니다.
- `allow_dangerous_deserialization=True` : FAISS의 `index.pkl` 은 파이썬 `pickle` 형식이라 로드 시 명시적 허용이 필요합니다. *직접 만든 신뢰할 수 있는 파일* 일 때만 사용하세요.

In [3]:
import time

from langchain_voyageai import VoyageAIEmbeddings
from langchain_community.vectorstores import FAISS

class RateLimitedVoyageEmbeddings(VoyageAIEmbeddings):
    """무료 한도(3 RPM / 10K TPM) 초과 시 잠시 대기 후 자동 재시도하는 임베딩.

    RateLimitError 가 나면 25초 기다렸다가 다시 시도하므로,
    빌드·검색 어디서 호출되든 한도에 걸려도 알아서 통과한다.
    (근본 해결은 결제수단 등록 — 무료 토큰은 그대로 적용된다.)
    """

    def _with_retry(self, fn, arg):
        for attempt in range(6):
            try:
                return fn(arg)
            except Exception as e:
                if "RateLimit" not in type(e).__name__:
                    raise
                print(f"  [rate limit] 25초 대기 후 재시도 ({attempt + 1}/6)")
                time.sleep(25)
        raise RuntimeError("재시도 초과 - 잠시 후 다시 실행하거나 결제수단을 등록하세요.")

    def embed_documents(self, texts):
        return self._with_retry(super().embed_documents, texts)

    def embed_query(self, text):
        return self._with_retry(super().embed_query, text)


embeddings = RateLimitedVoyageEmbeddings(model=os.getenv("VOYAGE_EMBEDDING_MODEL"))

In [4]:
vectorstore = FAISS.load_local(
    "../data/vector_db/osh_act_faiss",
    embeddings,
    allow_dangerous_deserialization=True,
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print("벡터 DB 로드 완료")

벡터 DB 로드 완료


### 2. RAG 프롬프트 설계 ⭐

RAG에서 프롬프트는 단순한 질문이 아니라 **"검색된 문서를 어떻게 다룰지"에 대한 규칙서** 입니다. 잘 설계된 프롬프트는 다음 요소를 담습니다.

- **페르소나** : 어떤 역할로 답할지 (산업안전보건법 안내 도우미)
- **근거 제한(그라운딩)** : *주어진 참고문서의 내용만* 으로 답하라 — 모델의 사전지식이나 추측을 막는다
- **모름 처리** : 문서에 근거가 없으면 "찾을 수 없다"고 솔직히 답하라 — RAG에서 환각을 막는 **가장 중요한 장치**
- **출력 형식** : 한국어로, 가능하면 **근거 조항** 을 함께 제시

`{context}` 에는 검색된 문서가, `{question}` 에는 사용자의 질문이 채워집니다.

In [5]:
from langchain_core.prompts import PromptTemplate

rag_prompt = PromptTemplate.from_template(
    """너는 산업안전보건법을 안내하는 친절한 법령 도우미야.

아래 #참고문서 의 내용만을 근거로 사용자의 질문에 답해.

[규칙]
- 반드시 #참고문서 안에 있는 내용만으로 답한다. 너의 사전지식이나 추측으로 답하지 않는다.
- 참고문서에서 답의 근거를 찾을 수 없으면, "제공된 문서에서 해당 내용을 찾을 수 없습니다." 라고 답한다.
- 한국어로, 핵심을 먼저 간결하게 답한 뒤 필요하면 부연한다.
- 가능하면 근거가 되는 조항(예: 제2조)을 함께 밝힌다.

#참고문서:
{context}

#질문:
{question}

#답변:"""
)

### 3. RAG 함수 만들기 (검색 → 프롬프트 → LLM)

이제 RAG 흐름을 **평범한 파이썬 함수** 하나로 만듭니다. 특별한 연결 문법 없이, 4단계를 함수 안에서 **한 줄씩 차례로** 실행하므로 흐름이 그대로 눈에 보입니다.

> ① 검색(`retriever.invoke`) → ② 프롬프트 채우기(`prompt.format`) → ③ LLM 호출(`llm.invoke`) → ④ 답변(`.content`)

- `format_docs` : 검색된 여러 청크(`Document` 리스트)를 **하나의 문자열** 로 합쳐 `{context}` 자리에 넣기 위한 헬퍼입니다.
- `temperature=0` : 사실을 다루는 법령 답변이므로 **일관되고 보수적인** 출력을 위해 0으로 둡니다.
- `prompt` 를 인자로 받게 해 두면, 같은 함수로 **여러 프롬프트를 갈아 끼우며** 비교할 수 있습니다(다음 섹션에서 활용).

In [6]:
from langchain.chat_models import init_chat_model

llm = init_chat_model(model="claude-haiku-4-5", temperature=0)


def format_docs(docs):
    """검색된 문서들을 하나의 문자열로 합친다."""
    return "\n\n".join(doc.page_content for doc in docs)


def ask_rag(question, prompt=rag_prompt):
    """RAG 4단계를 순서대로 직접 수행한다: 검색 → 프롬프트 → LLM → 답변."""
    # ① 검색: 질문과 관련된 문서를 가져온다
    docs = retriever.invoke(question)

    # ② 프롬프트: 검색한 문서를 context 자리에 채워 넣는다
    filled_prompt = prompt.format(context=format_docs(docs), question=question)

    # ③ LLM: 완성된 프롬프트로 답을 생성한다
    response = llm.invoke(filled_prompt)

    # ④ 답변 텍스트만 돌려준다
    return response.content

체인에 법령 질문을 넣어 실행해 봅니다.

In [7]:
print(ask_rag("중대재해란 무엇인가요?"))

제공된 문서에서 해당 내용을 찾을 수 없습니다.

참고문서에는 "중대재해"라는 용어가 여러 번 등장하지만, 중대재해의 정의나 구체적인 개념을 설명하는 조항이 포함되어 있지 않습니다. 중대재해의 정의를 알기 위해서는 산업안전보건법의 제2조(정의) 등 다른 부분을 확인하실 필요가 있습니다.


In [8]:
for question in ["근로자의 의무는 무엇인가요?", "도급인의 정의를 알려줘"]:
    print(f"Q. {question}")
    print(ask_rag(question))
    print("=" * 70)

Q. 근로자의 의무는 무엇인가요?
# 근로자의 의무

제6조에 따르면, 근로자의 의무는 다음과 같습니다:

## 핵심 의무
1. **산업재해 예방 기준 준수** - 산업안전보건법과 이 법에 따른 명령으로 정하는 산업재해 예방을 위한 기준을 지켜야 합니다.

2. **예방 조치 이행** - 사업주 또는 노동감독관, 공단 등 관계인이 실시하는 산업재해 예방에 관한 조치에 따라야 합니다.

## 추가 권리
또한 제52조에 따르면, 근로자는 **산업재해가 발생할 급박한 위험이 있는 경우에는 작업을 중지하고 대피할 수 있으며**, 이 경우 지체 없이 그 사실을 관리감독자 등에게 보고해야 합니다.
Q. 도급인의 정의를 알려줘
# 도급인의 정의

**도급인**은 물건의 제조·건설·수리 또는 서비스의 제공, 그 밖의 업무를 도급하는 사업주를 말합니다. 다만, 건설공사발주자는 제외됩니다.

(제2조 제7호)

**부연:**
- 도급인은 다른 사업주에게 일을 맡기는 입장의 사업주입니다.
- 건설공사발주자(건설공사를 도급하지만 시공을 주도하여 총괄·관리하지 않는 자)는 도급인에서 제외됩니다.


### 4. 프롬프트의 힘 — 그라운딩 비교 ⭐

RAG에서 프롬프트가 왜 중요한지 직접 확인해 봅니다. **같은 검색 결과** 를 주더라도, 프롬프트에 따라 답의 신뢰도가 크게 달라집니다.

- **나쁜 프롬프트** : 그저 "문서를 참고해 답해" 정도. 모델이 문서를 벗어나 사전지식으로 *그럴듯하게 지어낼* 여지가 있다.
- **좋은 프롬프트** : 위 2번에서 만든 것. "문서에 없으면 모른다고 답하라"는 **그라운딩 규칙** 이 들어 있다.

특히 **문서에 없는 질문** 을 던지면 차이가 분명해집니다. 산업안전보건법에는 "연차 휴가" 규정이 없으므로(이는 근로기준법 영역), 좋은 프롬프트는 "찾을 수 없다"고 답해야 합니다.

In [9]:
# 그라운딩 규칙이 없는 '나쁜' 프롬프트
weak_prompt = PromptTemplate.from_template(
    """다음 문서를 참고해서 질문에 답해줘.

문서:
{context}

질문: {question}
답변:"""
)

In [10]:
def compare(question):
    """같은 질문을 '나쁜 프롬프트'와 '좋은 프롬프트'로 각각 답하게 해 비교한다."""
    print(f"질문: {question}\n")
    print("[나쁜 프롬프트 — 그라운딩 규칙 없음]")
    print(ask_rag(question, weak_prompt))
    print("\n" + "-" * 70 + "\n")
    print("[좋은 프롬프트 — 문서에 없으면 모른다고 답]")
    print(ask_rag(question, rag_prompt))
    print("=" * 70)

In [11]:
# 산업안전보건법 범위 밖 질문 → 좋은 프롬프트는 '찾을 수 없음'으로 답해야 한다
compare("근로자의 연차 휴가 일수는 며칠인가요?")

질문: 근로자의 연차 휴가 일수는 며칠인가요?

[나쁜 프롬프트 — 그라운딩 규칙 없음]
  [rate limit] 25초 대기 후 재시도 (1/6)
죄송하지만, 제공하신 문서에는 근로자의 연차 휴가 일수에 관한 내용이 포함되어 있지 않습니다.

문서는 「산업안전보건법」의 내용으로, 주로 다음과 같은 사항들을 다루고 있습니다:
- 근로자의 작업중지 및 대피 권리
- 사업주의 안전 및 보건 조치 의무
- 고객응대근로자 보호
- 유해위험방지계획서 작성

연차 휴가에 관한 규정은 「근로기준법」에서 정하고 있으므로, 해당 법률을 참고하셔야 정확한 답변을 얻을 수 있습니다.

----------------------------------------------------------------------

[좋은 프롬프트 — 문서에 없으면 모른다고 답]
제공된 문서에서 해당 내용을 찾을 수 없습니다.

참고하신 문서는 산업안전보건법의 일부 조항들로, 주로 작업장의 안전 및 보건 조치, 근로자의 작업중지권, 유해위험방지계획서 등에 관한 내용입니다. 연차 휴가에 관한 사항은 포함되어 있지 않습니다.

연차 휴가 관련 규정은 「근로기준법」에서 다루고 있습니다.


> **핵심**: 검색 결과가 같아도 **프롬프트의 그라운딩 규칙** 하나로 환각 여부가 갈립니다. RAG에서 프롬프트는 검색만큼 중요합니다.

### 5. 출처(근거) 함께 보여주기

RAG의 큰 장점은 **답변의 근거를 제시** 할 수 있다는 점입니다. 사용자가 답을 검증할 수 있어 신뢰도가 올라갑니다.

함수 방식에서는 아주 간단합니다. 함수가 검색 결과 `docs` 를 **이미 손에 쥐고 있으니**, 답변과 함께 그대로 돌려주기만 하면 됩니다.

In [12]:
def ask_rag_with_source(question, prompt=rag_prompt):
    """답변과 함께, 그 답변의 근거가 된 문서(docs)도 같이 돌려준다."""
    docs = retriever.invoke(question)
    filled_prompt = prompt.format(context=format_docs(docs), question=question)
    answer = llm.invoke(filled_prompt).content
    return answer, docs

In [13]:
answer, docs = ask_rag_with_source("사업주의 의무는 무엇인가요?")

print("[답변]")
print(answer)

print("\n[근거 문서]")
for doc in docs:
    page = doc.metadata.get("page")
    print(f"- (p.{page}) {doc.page_content[:100]} ...")

[답변]
# 사업주의 의무

산업안전보건법에서 규정하는 사업주의 주요 의무는 다음과 같습니다. (제5조)

## 1. 근로자의 안전 및 건강 유지·증진
사업주는 다음 사항을 이행하여 근로자의 안전 및 건강을 유지·증진시키고 국가의 산업재해 예방정책을 따라야 합니다:

- **산업재해 예방 기준 준수**: 이 법과 이에 따른 명령으로 정하는 산업재해 예방 기준 준수
- **쾌적한 작업환경 조성**: 근로자의 신체적 피로와 정신적 스트레스 등을 줄일 수 있는 쾌적한 작업환경 조성 및 근로조건 개선
- **정보 제공**: 해당 사업장의 안전 및 보건에 관한 정보를 근로자에게 제공

## 2. 발주·설계·제조·수입·건설 시 의무
기계·기구, 설비, 원재료 등을 발주·설계·제조·수입 또는 건설할 때 관련 기준을 지키고, 이로 인한 산업재해를 방지하기 위한 필요한 조치를 해야 합니다.

[근거 문서]
- (p.1) 법제처                                                            2                                     ...
- (p.4) 법제처                                                            5                                     ...
- (p.5) 법제처                                                            6                                     ...


### 6. (선택) 구조화된 출력으로 받기

답변을 **정해진 형식** 으로 안정적으로 받고 싶을 때가 있습니다(예: 후속 시스템에 전달). `1.2` 에서 다룬 **structured output** 을 RAG에 적용해 봅니다.

`grounded` 필드로 "문서에서 근거를 찾았는지" 까지 모델이 스스로 표시하게 하면, 환각 여부를 프로그램적으로 다룰 수 있습니다.

In [14]:
from pydantic import BaseModel, Field


class LawAnswer(BaseModel):
    """산업안전보건법 질의응답 결과"""

    answer: str = Field(description="질문에 대한 답변")
    source_pages: list[int] = Field(description="근거가 된 참고문서의 페이지 번호 목록")
    grounded: bool = Field(description="참고문서에서 근거를 찾았으면 True, 못 찾았으면 False")

In [15]:
from langchain_core.messages import HumanMessage, SystemMessage

structured_llm = llm.with_structured_output(LawAnswer)


def ask_structured(question: str) -> LawAnswer:
    docs = retriever.invoke(question)
    # 페이지 번호를 함께 보여주어 모델이 source_pages 를 채우도록 돕는다
    context = "\n\n".join(
        f"(page {doc.metadata.get('page')}) {doc.page_content}" for doc in docs
    )
    return structured_llm.invoke(
        [
            SystemMessage(content=rag_prompt.format(context=context, question=question)),
            HumanMessage(content=question),
        ]
    )

In [16]:
res = ask_structured("중대재해란 무엇인가요?")
print("grounded     :", res.grounded)
print("source_pages :", res.source_pages)
print("answer       :", res.answer)

print("\n--- 문서 밖 질문 ---")
res2 = ask_structured("연차 휴가 일수는?")
print("grounded     :", res2.grounded)
print("answer       :", res2.answer)

  [rate limit] 25초 대기 후 재시도 (1/6)
  [rate limit] 25초 대기 후 재시도 (2/6)
grounded     : False
source_pages : [13, 14]
answer       : 제공된 문서에서 중대재해의 정의를 명시적으로 설명하는 내용을 찾을 수 없습니다. 

다만, 참고문서에서는 중대재해와 관련된 여러 규정들을 다루고 있습니다:
- 제54조에서 중대재해 발생 시 사업주의 조치 의무
- 제55조에서 중대재해 발생 시 고용노동부장관의 작업중지 조치
- 제56조에서 중대재해등의 원인조사

중대재해의 구체적인 정의와 범위를 알기 위해서는 산업안전보건법의 초반부 조항(예: 제2조 정의 부분)을 확인하실 필요가 있습니다.

--- 문서 밖 질문 ---
grounded     : False
answer       : 제공된 문서에서 해당 내용을 찾을 수 없습니다. 산업안전보건법 문서에는 연차 휴가 일수에 관한 규정이 포함되어 있지 않습니다. 연차 휴가는 근로기준법에서 규정하고 있습니다.


### 7. 하나의 함수로 묶기 — ask_law()

지금까지의 흐름(검색 → 답변 → 출처)을 **재사용 가능한 함수** 로 정리합니다. 실제 서비스에서는 이런 형태로 RAG를 호출하게 됩니다.

In [17]:
def ask_law(question):
    """산업안전보건법에 대해 질문하면 답변과 근거 페이지를 함께 출력한다."""
    answer, docs = ask_rag_with_source(question)

    print(f"Q. {question}\n")
    print(answer)

    pages = sorted({doc.metadata.get("page") for doc in docs})
    print(f"\n근거 페이지: {pages}")
    return answer, docs

In [18]:
_ = ask_law("안전보건진단이란 무엇인가요?")

Q. 안전보건진단이란 무엇인가요?

# 안전보건진단의 정의

**안전보건진단이란 산업재해를 예방하기 위하여 잠재적 위험성을 발견하고 그 개선대책을 수립할 목적으로 조사·평가하는 것입니다.** (제2조제12호)

## 부연 설명

안전보건진단은 다음과 같은 특징이 있습니다:

- **대상**: 추락·붕괴, 화재·폭발, 유해하거나 위험한 물질의 누출 등 산업재해 발생의 위험이 현저히 높은 사업장 (제47조제1항)

- **실시기관**: 고용노동부장관이 지정한 안전보건진단기관이 실시합니다 (제47조제1항, 제48조)

- **결과보고**: 안전보건진단기관은 진단 결과보고서를 사업주 및 고용노동부장관에게 제출해야 합니다 (제47조제4항)

근거 페이지: [0, 11, 12]


### 정리

- **RAG** 는 *검색(Retriever) → 프롬프트 → LLM* 흐름으로, LLM이 **건네받은 문서를 근거로** 답하게 해 환각을 줄인다.
- 저장해 둔 벡터 DB는 `FAISS.load_local` 로 불러와 바로 검색기로 쓸 수 있다.
- **프롬프트 설계가 핵심** 이다. 특히 *"문서에 없으면 모른다고 답하라"* 는 그라운딩 규칙이 RAG 신뢰성을 좌우한다.
- 답변과 함께 **출처(페이지·조항)** 를 제시하면 사용자가 검증할 수 있어 신뢰도가 올라간다.
- 구조화된 출력으로 받으면 후속 처리(환각 여부 판단 등)가 쉬워진다.

> **좋은 RAG = 좋은 검색 + 좋은 프롬프트(그라운딩 · 출처)**

### (참고) LCEL — 같은 흐름을 한 줄로 잇는 방식

LangChain에는 위 단계(검색 → 프롬프트 → LLM → 출력)를 `|` 기호로 이어 붙이는 **LCEL(LangChain Expression Language)** 표기법도 있습니다.

```python
# 아래는 참고용 — 이 노트북에서는 실행하지 않습니다.
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)
answer = rag_chain.invoke("중대재해란 무엇인가요?")
```

동작 결과는 위 `ask_rag()` 함수와 같습니다. 간결하지만 처음 배울 때는 흐름이 한눈에 들어오지 않으므로, 이 강의에서는 **단계가 그대로 보이는 함수 방식**을 사용했습니다.